# OLMoEarth Pixel Embeddings

Generate pixel embeddings from Sentinel-2 time series with the Pixelverse registry.

- input: `B, T, C, H, W`
- timestamps: `B, T, 3` (`[day, month, year]`, month zero-indexed)
- output: `B, H, W, D` (default `patch_size=1`)


In [1]:
import numpy as np
import rioxarray  # noqa: F401
import torch
import xarray as xr

import pixelverse as pv

In [2]:
MODEL_NAME = "olmoearth_nano"
# Use pretrained=False for a fast, fully local notebook run.
# Switch to pretrained=True for real inference.
model, transforms = pv.create_model(MODEL_NAME, pretrained=False)
weights = pv.get_weights(MODEL_NAME)
model.eval()
weights.meta

{'model_id': 'OlmoEarth-v1-Nano',
 'bands': ['B02',
  'B03',
  'B04',
  'B08',
  'B05',
  'B06',
  'B07',
  'B8A',
  'B11',
  'B12',
  'B01',
  'B09'],
 's2_asset_names': ['coastal',
  'blue',
  'green',
  'red',
  'rededge1',
  'rededge2',
  'rededge3',
  'nir',
  'nir08',
  'nir09',
  'swir16',
  'swir22'],
 'num_channels': 12,
 'embed_dim': 128,
 'input_shape': [(None, None, 12, None, None)],
 'timestamps_shape': [(None, None, 3)],
 'output_shape': [(None, None, None, 128)],
 'patch_size': 1,
 'input_res': 10,
 'variant': 'olmoearth_nano'}

## Sentinel-2 Band Order

Required channel order (`C`):

`B02, B03, B04, B08, B05, B06, B07, B8A, B11, B12, B01, B09`


In [3]:
OLMOEARTH_BANDS = [
    "B02",
    "B03",
    "B04",
    "B08",
    "B05",
    "B06",
    "B07",
    "B8A",
    "B11",
    "B12",
    "B01",
    "B09",
]

S2_COMMON_NAME_TO_OLMOEARTH = {
    "coastal": "B01",
    "blue": "B02",
    "green": "B03",
    "red": "B04",
    "nir": "B08",
    "rededge1": "B05",
    "rededge2": "B06",
    "rededge3": "B07",
    "nir08": "B8A",
    "swir16": "B11",
    "swir22": "B12",
    "nir09": "B09",
}


def build_olmoearth_inputs_from_xarray(ds: xr.Dataset) -> tuple[torch.Tensor, torch.Tensor]:
    if set(S2_COMMON_NAME_TO_OLMOEARTH).issubset(ds.data_vars):
        ds = ds.rename(S2_COMMON_NAME_TO_OLMOEARTH)

    missing = [band for band in OLMOEARTH_BANDS if band not in ds]
    if missing:
        raise KeyError(f"Dataset is missing required bands: {missing}")

    arr = ds[OLMOEARTH_BANDS].to_array(dim="band").transpose("time", "band", "y", "x").values
    x = torch.from_numpy(arr).unsqueeze(0).float()  # (B, T, C, H, W)

    t = ds.time.dt
    timestamps_np = np.stack(
        [
            t.day.values.astype(np.int64),
            (t.month.values - 1).astype(np.int64),
            t.year.values.astype(np.int64),
        ],
        axis=-1,
    )
    timestamps = torch.from_numpy(timestamps_np).unsqueeze(0)
    return x, timestamps

## Synthetic Example

Runs end-to-end without downloading STAC data.


In [4]:
B, T, C, H, W = 1, 4, 12, 16, 16
x = torch.randint(0, 10000, (B, T, C, H, W), dtype=torch.int32).float()
timestamps = torch.tensor(
    [[[15, 0, 2024], [15, 1, 2024], [15, 2, 2024], [15, 3, 2024]]],
    dtype=torch.long,
)

with torch.no_grad():
    embeddings = model(transforms(x), timestamps=timestamps)

embeddings.shape  # (B, H, W, D) with patch_size=1

torch.Size([1, 16, 16, 128])

## Convert To Xarray

Optional feature-first view: `(feature, y, x)`.


In [5]:
embeddings_da = xr.DataArray(
    embeddings[0].cpu().numpy(),
    dims=["y", "x", "feature"],
).transpose("feature", "y", "x")
embeddings_da

<xarray.DataArray (feature: 128, y: 16, x: 16)> Size: 131kB
array([[[-5.98711073e-01, -5.91792107e-01, -6.49102271e-01, ...,
         -6.04611456e-01, -6.24796331e-01, -6.20052218e-01],
        [-6.74395025e-01, -7.18417406e-01, -6.89917505e-01, ...,
         -7.07893610e-01, -7.25913048e-01, -7.24643767e-01],
        [-5.46095788e-01, -6.02966547e-01, -5.77718973e-01, ...,
         -6.91933215e-01, -6.22837365e-01, -6.24790728e-01],
        ...,
        [-8.21984351e-01, -7.13532507e-01, -8.03522825e-01, ...,
         -8.30226660e-01, -6.54963136e-01, -7.32480228e-01],
        [-7.83734143e-01, -8.41842353e-01, -7.90852726e-01, ...,
         -9.25392151e-01, -7.98426449e-01, -8.20673943e-01],
        [-7.87093699e-01, -6.03030205e-01, -7.64714897e-01, ...,
         -7.21225441e-01, -7.05101073e-01, -7.70181894e-01]],

       [[ 1.67562738e-01,  3.39913994e-01,  2.61052161e-01, ...,
          5.49960732e-02, -6.11905940e-02,  8.76080692e-02],
        [ 3.00992578e-01,  2.58007079e-01,  1.30919769e-01, ...,
         -7.23864213e-02, -9.76831391e-02,  8.37294683e-02],
        [ 1.97840199e-01,  2.00124025e-01,  1.48115292e-01, ...,
          1.32264674e-01, -3.39230783e-02,  9.65773761e-02],
...
          5.50359488e-01,  2.70646065e-01,  2.70414025e-01],
        [ 2.53034860e-01,  2.01784313e-01,  1.35546774e-01, ...,
          3.10440809e-01,  2.49061361e-01,  3.47402900e-01],
        [ 3.46582562e-01,  1.80914089e-01,  2.57160038e-01, ...,
          2.18096927e-01,  2.43104205e-01,  2.48287097e-01]],

       [[-4.27795798e-01, -3.46989781e-01, -3.76844257e-01, ...,
         -2.74180412e-01, -2.87835091e-01, -3.85809749e-01],
        [-3.42460394e-01, -3.86223823e-01, -3.85417461e-01, ...,
         -2.55189151e-01, -4.45228696e-01, -1.81210473e-01],
        [-2.55297452e-01, -3.14825684e-01, -2.30060756e-01, ...,
         -3.00038785e-01, -3.69696498e-01, -3.31187904e-01],
        ...,
        [-3.67298216e-01, -3.69606525e-01, -3.83567899e-01, ...,
         -4.22587067e-01, -3.58290672e-01, -2.64165491e-01],
        [-3.78918439e-01, -3.36103886e-01, -4.73224968e-01, ...,
         -4.88374263e-01, -6.16900980e-01, -3.30492318e-01],
        [-5.06732762e-01, -2.99308866e-01, -2.78218478e-01, ...,
         -3.16872090e-01, -3.91126752e-01, -4.42824095e-01]]],
      shape=(128, 16, 16), dtype=float32)
Dimensions without coordinates: feature, y, x

## Example With `get_s2_time_series`

Use `weights.meta["s2_asset_names"]` for the download asset list.


In [6]:
# Example (not executed here):
# from pixelverse.download.sentinel2 import get_s2_time_series
#
# weights = pv.get_weights("olmoearth_nano")
# ds = get_s2_time_series(
#     bbox=...,
#     year=2024,
#     assets=weights.meta["s2_asset_names"],
# )
# x, timestamps = build_olmoearth_inputs_from_xarray(ds)
# with torch.no_grad():
#     embeddings = model(transforms(x), timestamps=timestamps)


## Save\n
\n
Convert to a dataset for COG/Zarr-style export examples.\n

In [6]:
embeddings_ds = embeddings_da.to_dataset(name="embedding")
embeddings_ds

<xarray.Dataset> Size: 131kB
Dimensions:    (feature: 128, y: 16, x: 16)
Dimensions without coordinates: feature, y, x
Data variables:
    embedding  (feature, y, x) float32 131kB -0.5987 -0.5918 ... -0.3911 -0.4428

In [7]:
# Save to Zarr (runnable on synthetic example)
embeddings_ds.to_zarr("olmoearth_pixel_embeddings.zarr", mode="w")

/Users/isaaccorley/.codex/worktrees/e7e8/pixelverse/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [9]:
# COG example (requires georeferenced coords/CRS, e.g. real raster/xarray input)
# embeddings_ds["embedding"].rio.to_raster(
#     "olmoearth_pixel_embeddings_cog.tif",
#     driver="COG",
#     compress="ZSTD",
# )


In [10]:
# Optional NumPy export
np.save("olmoearth_pixel_embeddings.npy", embeddings.cpu().numpy())